#### **Validation, `field_validator` and `model_validator`**

1. **Built-in validation**
2. **`field_validator()`** — validate one or more fields
3. **`model_validator()`** — validate the entire model

---

#### **What is Validation?**

Validation means checking whether data is acceptable according to the rules you've defined. For example:

In [1]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

# Pydantic automatically validate the data
user = User(name='Shourov Roy', age=23)
print(user)

name='Shourov Roy' age=23


But invalid data can cause a `ValidationError`:

```python
user = User(name="Shouro", age="hello")
```

Pydantic will report that `"hello"` cannot be converted to an integer.

You can also add your own custom validation rules. That's where `field_validator()` and `model_validator()` become useful.

---

#### **`field_validator()`**

`field_validator()` is used when you want to validate or modify a **specific field**.

In [3]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    username: str
    age: int

    @field_validator("username")
    @classmethod
    def validate_username(cls, value):
        if len(value) < 3:
            raise ValueError("Username must contain at least 3 characters.")
        return value

user1 = User(username="Shourov Roy", age=23)
print(user1)

# This raise error
# user2 = User(username="Sk", age=23)
# print(user2)

username='Shourov Roy' age=23


---

#### **Why do we use `@classmethod`?**

```python
@field_validator("username")
@classmethod
def validate_username(cls, value):
    ...
```

The validator is defined as a class method because Pydantic uses it as part of the model's validation system.

---
**Important: Always Return the Value:** A validator generally needs to return the value that should be stored. You can also modify the value:

In [6]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    username: str
    age: int

    @field_validator("username")
    @classmethod
    def clean_username(cls, value):
        return value.strip().lower()

user = User(username="  Shourov  ", age=23)
print(user.username)

shourov


So validators can perform both:

* **validation**
* **normalization/transformation**

---

**`mode="before"`:** By default, a field validator runs after Pydantic has performed its normal validation. You can change this using: `mode="before"`. Example:

```python
from pydantic import BaseModel, field_validator

class User(BaseModel):
    age: int

    @field_validator("age", mode="before")
    @classmethod
    def validate_age(cls, value):
        print("Before:", value)
        return value
```

Here the validator receives the **raw input** before Pydantic converts it to the declared type.

---
#### **When is `mode="before"` useful?**

It is useful when you want to:

* clean raw input
* transform input
* handle different input formats
* preprocess values before type validation

---

#### **`model_validator()`**

Sometimes validation requires looking at **multiple fields together**. For example, suppose a user must provide:

```text
password
confirm_password
```

We need to compare them. This isn't really a single-field validation problem. We need **model-level validation**. Use:

```python
model_validator()
```

In [9]:
from pydantic import BaseModel, model_validator

class User(BaseModel):
    username: str
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def check_passwords(self):
        if self.password != self.confirm_password:
            raise ValueError("Password do not match")
        return self

# This work but changing password is not work
user = User(username="Shourov", password="123", confirm_password="123")
print(user)

username='Shourov' password='123' confirm_password='123'


---

#### **Field Validator vs Model Validator**

| Validator                             | Purpose                                  |
| ------------------------------------- | ---------------------------------------- |
| `field_validator()`                   | Validate a field                         |
| `model_validator()`                   | Validate the entire model                |
| `field_validator(..., mode="before")` | Process a field before normal validation |
| `field_validator(..., mode="after")`  | Validate a field after normal validation |
| `model_validator(mode="before")`      | Process raw model input                  |
| `model_validator(mode="after")`       | Validate the completed model             |


---

#### Complete Example

In [10]:
from pydantic import BaseModel, field_validator, model_validator

class UserRegistration(BaseModel):
    username: str
    email: str
    age: int
    password: str
    confirm_password: str

    @field_validator("username")
    @classmethod
    def validate_username(cls, value):
        value = value.strip()
        if len(value) < 3:
            raise ValueError("Username must be at least 3 characters")
        return value

    @field_validator("email")
    @classmethod
    def validate_email(cls, value):
        value = value.strip().lower()
        if "@" not in value:
            raise ValueError("Invalid email address")
        return value

    @field_validator("age")
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("User must be at least 18")
        return value

    @model_validator(mode="after")
    def validate_passwords(self):
        if self.password != self.confirm_password:
            raise ValueError("Password do not match")
        return self

user = UserRegistration(
    username=" Shourov Roy ",
    email="Shourov@example.com",
    age=23,
    password="123",
    confirm_password="123"
)
print(user)

username='Shourov Roy' email='shourov@example.com' age=23 password='123' confirm_password='123'
